In [35]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
# from sklearn.model_selection

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)

In [36]:
df = pd.read_csv('balanced_dataset_12k.csv')

In [37]:
df.head()

,headline,label,confidence
0,"How JD(S) is quietly remaking its leadership team: As a veteran MLA is removed from core committee, ‘first family’ scions get key posts",sarcastic,0.65
1,"Waqf Bill: More than 12 hours of debate, and a washroom break that got Congress all wet",sarcastic,0.85
2,"India-U.S. interim trade deal: ‘Namaste Trump scored over Howdy Modi’, says Congress",sarcastic,0.75
3,"‘Richest’ CM Chandrababu Naidu’s draws almost entirely on one asset, Mamata records sharpest decline",sarcastic,0.45
4,Sunil Jakhar ‘resignation’: How ex-Congress leader struggled to find feet in BJP,sarcastic,0.45


In [38]:
df = df.drop(columns=['confidence'], axis = 1)

In [39]:
df['clean_text'] = df['headline'].str.lower()
df.head()

,headline,label,clean_text
0,"How JD(S) is quietly remaking its leadership team: As a veteran MLA is removed from core committee, ‘first family’ scions get key posts",sarcastic,"how jd(s) is quietly remaking its leadership team: as a veteran mla is removed from core committee, ‘first family’ scions get key posts"
1,"Waqf Bill: More than 12 hours of debate, and a washroom break that got Congress all wet",sarcastic,"waqf bill: more than 12 hours of debate, and a washroom break that got congress all wet"
2,"India-U.S. interim trade deal: ‘Namaste Trump scored over Howdy Modi’, says Congress",sarcastic,"india-u.s. interim trade deal: ‘namaste trump scored over howdy modi’, says congress"
3,"‘Richest’ CM Chandrababu Naidu’s draws almost entirely on one asset, Mamata records sharpest decline",sarcastic,"‘richest’ cm chandrababu naidu’s draws almost entirely on one asset, mamata records sharpest decline"
4,Sunil Jakhar ‘resignation’: How ex-Congress leader struggled to find feet in BJP,sarcastic,sunil jakhar ‘resignation’: how ex-congress leader struggled to find feet in bjp


In [40]:
import string
def remove_punctuations(text):
    punctuations = string.punctuation
    return text.translate(str.maketrans('', '', punctuations))

df['clean_text'] = df['clean_text'].apply(lambda x: remove_punctuations(x))
df.head()

,headline,label,clean_text
0,"How JD(S) is quietly remaking its leadership team: As a veteran MLA is removed from core committee, ‘first family’ scions get key posts",sarcastic,how jds is quietly remaking its leadership team as a veteran mla is removed from core committee ‘first family’ scions get key posts
1,"Waqf Bill: More than 12 hours of debate, and a washroom break that got Congress all wet",sarcastic,waqf bill more than 12 hours of debate and a washroom break that got congress all wet
2,"India-U.S. interim trade deal: ‘Namaste Trump scored over Howdy Modi’, says Congress",sarcastic,indiaus interim trade deal ‘namaste trump scored over howdy modi’ says congress
3,"‘Richest’ CM Chandrababu Naidu’s draws almost entirely on one asset, Mamata records sharpest decline",sarcastic,‘richest’ cm chandrababu naidu’s draws almost entirely on one asset mamata records sharpest decline
4,Sunil Jakhar ‘resignation’: How ex-Congress leader struggled to find feet in BJP,sarcastic,sunil jakhar ‘resignation’ how excongress leader struggled to find feet in bjp


In [41]:
import nltk

nltk.download('stopwords')
nltk.download('punkt')

[nltk_data] Downloading package stopwords to
[nltk_data]     /home/jupyter/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /home/jupyter/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [42]:
from nltk.corpus import stopwords
", ".join(stopwords.words('english'))

"a, about, above, after, again, against, ain, all, am, an, and, any, are, aren, aren't, as, at, be, because, been, before, being, below, between, both, but, by, can, couldn, couldn't, d, did, didn, didn't, do, does, doesn, doesn't, doing, don, don't, down, during, each, few, for, from, further, had, hadn, hadn't, has, hasn, hasn't, have, haven, haven't, having, he, he'd, he'll, her, here, hers, herself, he's, him, himself, his, how, i, i'd, if, i'll, i'm, in, into, is, isn, isn't, it, it'd, it'll, it's, its, itself, i've, just, ll, m, ma, me, mightn, mightn't, more, most, mustn, mustn't, my, myself, needn, needn't, no, nor, not, now, o, of, off, on, once, only, or, other, our, ours, ourselves, out, over, own, re, s, same, shan, shan't, she, she'd, she'll, she's, should, shouldn, shouldn't, should've, so, some, such, t, than, that, that'll, the, their, theirs, them, themselves, then, there, these, they, they'd, they'll, they're, they've, this, those, through, to, too, under, until, up, 

In [43]:
STOPWORDS = set(stopwords.words('english'))
def remove_stopwords(text):
    return " ".join([word for word in text.split() if word not in STOPWORDS])

df['clean_text'] = df['clean_text'].apply(lambda x: remove_stopwords(x))
df.head()

,headline,label,clean_text
0,"How JD(S) is quietly remaking its leadership team: As a veteran MLA is removed from core committee, ‘first family’ scions get key posts",sarcastic,jds quietly remaking leadership team veteran mla removed core committee ‘first family’ scions get key posts
1,"Waqf Bill: More than 12 hours of debate, and a washroom break that got Congress all wet",sarcastic,waqf bill 12 hours debate washroom break got congress wet
2,"India-U.S. interim trade deal: ‘Namaste Trump scored over Howdy Modi’, says Congress",sarcastic,indiaus interim trade deal ‘namaste trump scored howdy modi’ says congress
3,"‘Richest’ CM Chandrababu Naidu’s draws almost entirely on one asset, Mamata records sharpest decline",sarcastic,‘richest’ cm chandrababu naidu’s draws almost entirely one asset mamata records sharpest decline
4,Sunil Jakhar ‘resignation’: How ex-Congress leader struggled to find feet in BJP,sarcastic,sunil jakhar ‘resignation’ excongress leader struggled find feet bjp


In [44]:
import re

def clean_text(text):
    # 1. Normalize weird unicode quotes → standard ones
    text = text.replace("’", "'").replace("‘", "'")
    text = text.replace("“", '"').replace("”", '"')

    # 2. Replace ellipsis with normal dots
    text = text.replace("…", "...")

    # 3. Remove unwanted symbols (but KEEP ! ? ' ")
    text = re.sub(r"[^a-zA-Z0-9\s!?\'\"]", " ", text)

    # 4. Remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text

df['clean_text'] = df['clean_text'].apply(lambda x: clean_text(x))
df.head()

,headline,label,clean_text
0,"How JD(S) is quietly remaking its leadership team: As a veteran MLA is removed from core committee, ‘first family’ scions get key posts",sarcastic,jds quietly remaking leadership team veteran mla removed core committee 'first family' scions get key posts
1,"Waqf Bill: More than 12 hours of debate, and a washroom break that got Congress all wet",sarcastic,waqf bill 12 hours debate washroom break got congress wet
2,"India-U.S. interim trade deal: ‘Namaste Trump scored over Howdy Modi’, says Congress",sarcastic,indiaus interim trade deal 'namaste trump scored howdy modi' says congress
3,"‘Richest’ CM Chandrababu Naidu’s draws almost entirely on one asset, Mamata records sharpest decline",sarcastic,'richest' cm chandrababu naidu's draws almost entirely one asset mamata records sharpest decline
4,Sunil Jakhar ‘resignation’: How ex-Congress leader struggled to find feet in BJP,sarcastic,sunil jakhar 'resignation' excongress leader struggled find feet bjp


In [45]:
!pip install nltk

In [46]:
from nltk.tokenize import TweetTokenizer

tokenizer = TweetTokenizer()
df['clean_text'] = df['clean_text'].apply(lambda x: tokenizer.tokenize(x))
df.head()

,headline,label,clean_text
0,"How JD(S) is quietly remaking its leadership team: As a veteran MLA is removed from core committee, ‘first family’ scions get key posts",sarcastic,"[jds, quietly, remaking, leadership, team, veteran, mla, removed, core, committee, ', first, family, ', scions, get, key, posts]"
1,"Waqf Bill: More than 12 hours of debate, and a washroom break that got Congress all wet",sarcastic,"[waqf, bill, 12, hours, debate, washroom, break, got, congress, wet]"
2,"India-U.S. interim trade deal: ‘Namaste Trump scored over Howdy Modi’, says Congress",sarcastic,"[indiaus, interim, trade, deal, ', namaste, trump, scored, howdy, modi, ', says, congress]"
3,"‘Richest’ CM Chandrababu Naidu’s draws almost entirely on one asset, Mamata records sharpest decline",sarcastic,"[', richest, ', cm, chandrababu, naidu's, draws, almost, entirely, one, asset, mamata, records, sharpest, decline]"
4,Sunil Jakhar ‘resignation’: How ex-Congress leader struggled to find feet in BJP,sarcastic,"[sunil, jakhar, ', resignation, ', excongress, leader, struggled, find, feet, bjp]"


In [47]:
df['text_for_bow'] = df['clean_text'].apply(
    lambda tokens: " ".join([word for word in tokens if word != "'"])
)

In [48]:
df.head()

,headline,label,clean_text,text_for_bow
0,"How JD(S) is quietly remaking its leadership team: As a veteran MLA is removed from core committee, ‘first family’ scions get key posts",sarcastic,"[jds, quietly, remaking, leadership, team, veteran, mla, removed, core, committee, ', first, family, ', scions, get, key, posts]",jds quietly remaking leadership team veteran mla removed core committee first family scions get key posts
1,"Waqf Bill: More than 12 hours of debate, and a washroom break that got Congress all wet",sarcastic,"[waqf, bill, 12, hours, debate, washroom, break, got, congress, wet]",waqf bill 12 hours debate washroom break got congress wet
2,"India-U.S. interim trade deal: ‘Namaste Trump scored over Howdy Modi’, says Congress",sarcastic,"[indiaus, interim, trade, deal, ', namaste, trump, scored, howdy, modi, ', says, congress]",indiaus interim trade deal namaste trump scored howdy modi says congress
3,"‘Richest’ CM Chandrababu Naidu’s draws almost entirely on one asset, Mamata records sharpest decline",sarcastic,"[', richest, ', cm, chandrababu, naidu's, draws, almost, entirely, one, asset, mamata, records, sharpest, decline]",richest cm chandrababu naidu's draws almost entirely one asset mamata records sharpest decline
4,Sunil Jakhar ‘resignation’: How ex-Congress leader struggled to find feet in BJP,sarcastic,"[sunil, jakhar, ', resignation, ', excongress, leader, struggled, find, feet, bjp]",sunil jakhar resignation excongress leader struggled find feet bjp


In [49]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer()

X = vectorizer.fit_transform(df['text_for_bow'])

y = df['label']

In [50]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,   # 30% goes to temp (val + test)
    random_state=42,
    stratify=y        # VERY IMPORTANT for classification
)

In [51]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=100,
    criterion='gini',
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    min_weight_fraction_leaf=0.0,
    max_features='sqrt',
    max_leaf_nodes=None,
    min_impurity_decrease=0.0,
    bootstrap=True,
    oob_score=False,
    n_jobs=-1,              # use all cores (recommended)
    random_state=42,
    verbose=0,
    warm_start=False,
    class_weight=None,
    ccp_alpha=0.0,
    max_samples=None,
    monotonic_cst=None
)

rf.fit(X_train, y_train)

,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [52]:
y_test_pred = rf.predict(X_test)

In [53]:
from sklearn.metrics import accuracy_score, classification_report

print("Validation Accuracy:", accuracy_score(y_test, y_test_pred))
print(classification_report(y_test, y_test_pred))

Validation Accuracy: 0.6601612218922359
               precision    recall  f1-score   support

non-sarcastic       0.69      0.58      0.63      1179
    sarcastic       0.64      0.74      0.68      1178

     accuracy                           0.66      2357
    macro avg       0.66      0.66      0.66      2357
 weighted avg       0.66      0.66      0.66      2357



In [54]:
from sklearn.svm import SVC

svm = SVC(
    C=1.0,
    kernel='linear',
    degree=3,
    gamma='scale',
    coef0=0.0,
    shrinking=True,
    probability=False,
    tol=0.001,
    cache_size=200,
    class_weight=None,
    verbose=False,
    max_iter=-1,
    decision_function_shape='ovr',
    break_ties=False,
    random_state=42
)

svm.fit(X_train, y_train)

,C,1.0
,kernel,'linear'
,degree,3
,gamma,'scale'
,coef0,0.0
,shrinking,True
,probability,False
,tol,0.001
,cache_size,200
,class_weight,None
,verbose,False


In [55]:
y_test_pred = svm.predict(X_test)

In [56]:
print("Validation Accuracy:", accuracy_score(y_test, y_test_pred))
print(classification_report(y_test, y_test_pred))

Validation Accuracy: 0.6342808655070005
               precision    recall  f1-score   support

non-sarcastic       0.63      0.64      0.64      1179
    sarcastic       0.64      0.62      0.63      1178

     accuracy                           0.63      2357
    macro avg       0.63      0.63      0.63      2357
 weighted avg       0.63      0.63      0.63      2357



In [57]:
from sklearn.experimental import enable_halving_search_cv
from sklearn.model_selection import HalvingGridSearchCV

param_grid = {
    'n_estimators': [100, 200, 300, 500],
    
    'max_depth': [None, 10, 20, 30, 50],
    
    'min_samples_split': [2, 5, 10],
    
    'min_samples_leaf': [1, 2, 4],
    
    'max_features': ['sqrt', 'log2', None],
    
    'bootstrap': [True, False],
    
    'criterion': ['gini', 'entropy']
}

from sklearn.experimental import enable_halving_search_cv
from sklearn.model_selection import HalvingGridSearchCV

from sklearn.experimental import enable_halving_search_cv
from sklearn.model_selection import HalvingGridSearchCV, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

param_grid = {
    # CORE (most impact)
    'n_estimators': [100, 200, 300],
    'max_depth': [10, 30, None],
    'max_features': ['sqrt', 0.5],

    # REGULARIZATION (important but controlled)
    'min_samples_split': [2, 10],
    'min_samples_leaf': [1, 3],

    # LIGHT extras (small exploration only)
    'bootstrap': [True],
    'criterion': ['gini', 'log_loss']
}

halving = HalvingGridSearchCV(
    RandomForestClassifier(
        random_state=42,
        n_jobs=1
    ),
    param_grid,
    cv=cv,
    factor=3,
    scoring='f1_weighted',
    n_jobs=-1,
    verbose=3
)

halving.fit(X_train, y_train)

print("Best Params:", halving.best_params_)

n_iterations: 5
n_required_iterations: 5
n_possible_iterations: 5
min_resources_: 116
max_resources_: 9427
aggressive_elimination: False
factor: 3
----------
iter: 0
n_candidates: 144
n_resources: 116
Fitting 5 folds for each of 144 candidates, totalling 720 fits
[CV 2/5] END bootstrap=True, criterion=gini, max_depth=10, max_features=sqrt, min_samples_leaf=1, min_samples_split=2, n_estimators=200;, score=(train=0.989, test=0.577) total time=   0.4s
[CV 1/5] END bootstrap=True, criterion=gini, max_depth=10, max_features=sqrt, min_samples_leaf=1, min_samples_split=10, n_estimators=100;, score=(train=0.989, test=0.415) total time=   0.2s
[CV 4/5] END bootstrap=True, criterion=gini, max_depth=10, max_features=sqrt, min_samples_leaf=1, min_samples_split=10, n_estimators=100;, score=(train=1.000, test=0.318) total time=   0.2s
[CV 5/5] END bootstrap=True, criterion=gini, max_depth=10, max_features=sqrt, min_samples_leaf=1, min_samples_split=10, n_estimators=200;, score=(train=0.586, test=0.3

In [58]:
best_rf = halving.best_estimator_

y_test_pred = best_rf.predict(X_test)

In [59]:
print("Test Accuracy:", accuracy_score(y_test, y_test_pred))
print(classification_report(y_test, y_test_pred))

Test Accuracy: 0.6521001272804412
               precision    recall  f1-score   support

non-sarcastic       0.67      0.61      0.64      1179
    sarcastic       0.64      0.70      0.67      1178

     accuracy                           0.65      2357
    macro avg       0.65      0.65      0.65      2357
 weighted avg       0.65      0.65      0.65      2357



In [60]:
from sklearn.svm import LinearSVC

param_grid = {
    'C': np.logspace(-3, 2, 8),
    'class_weight': [None, 'balanced'],
    'tol': [1e-3, 1e-4, 1e-5],
    'max_iter': [5000, 10000]   # removed 1000 (too small)
}

halving1 = HalvingGridSearchCV(
    LinearSVC(),   # ✅ FIXED
    param_grid,
    cv=cv,
    factor=3,
    scoring='f1_weighted',
    n_jobs=-1,
    verbose=2
)

halving1.fit(X_train, y_train)

print("Best Params:", halving1.best_params_)

n_iterations: 5
n_required_iterations: 5
n_possible_iterations: 5
min_resources_: 116
max_resources_: 9427
aggressive_elimination: False
factor: 3
----------
iter: 0
n_candidates: 96
n_resources: 116
Fitting 5 folds for each of 96 candidates, totalling 480 fits
[CV 5/5] END bootstrap=True, criterion=gini, max_depth=10, max_features=0.5, min_samples_leaf=3, min_samples_split=10, n_estimators=100;, score=(train=0.763, test=0.635) total time=   1.2s
[CV 3/5] END bootstrap=True, criterion=gini, max_depth=30, max_features=0.5, min_samples_leaf=3, min_samples_split=2, n_estimators=100;, score=(train=0.748, test=0.582) total time=   1.6s
[CV 2/5] END bootstrap=True, criterion=log_loss, max_depth=None, max_features=0.5, min_samples_leaf=3, min_samples_split=2, n_estimators=100;, score=(train=0.801, test=0.634) total time=   1.6s
[CV 5/5] END bootstrap=True, criterion=log_loss, max_depth=None, max_features=0.5, min_samples_leaf=3, min_samples_split=10, n_estimators=100;, score=(train=0.799, tes

In [61]:
best_svm = halving1.best_estimator_

y_test_pred = best_svm.predict(X_test)

In [62]:
print("Test Accuracy:", accuracy_score(y_test, y_test_pred))
print(classification_report(y_test, y_test_pred))

Test Accuracy: 0.6546457361052185
               precision    recall  f1-score   support

non-sarcastic       0.65      0.66      0.66      1179
    sarcastic       0.66      0.65      0.65      1178

     accuracy                           0.65      2357
    macro avg       0.65      0.65      0.65      2357
 weighted avg       0.65      0.65      0.65      2357

[CV] END C=19.306977288832496, class_weight=balanced, max_iter=5000, tol=0.001; total time=   0.0s
[CV] END C=19.306977288832496, class_weight=balanced, max_iter=5000, tol=1e-05; total time=   0.0s
[CV] END C=19.306977288832496, class_weight=balanced, max_iter=10000, tol=0.001; total time=   0.0s
[CV] END C=19.306977288832496, class_weight=balanced, max_iter=10000, tol=1e-05; total time=   0.0s
[CV] END C=100.0, class_weight=None, max_iter=5000, tol=0.0001; total time=   0.0s
[CV] END C=100.0, class_weight=None, max_iter=5000, tol=1e-05; total time=   0.0s
[CV] END C=100.0, class_weight=None, max_iter=10000, tol=0.0001; total

In [63]:
import joblib

joblib.dump(best_rf, "rf_model.joblib")
joblib.dump(best_svm, "lsvm_model.joblib")

['lsvm_model.joblib']